<a href="https://colab.research.google.com/github/AlishbaMalik687-svg/AI-ML_internship_Adv_Task_3/blob/master/Adv_task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Libraries

In [ ]:
# basic libraries
import numpy as np
import pandas as pd
import os
import cv2

# deep learning
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input, concatenate
from tensorflow.keras.models import Model

# evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

Connecting with Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Giving Dataset path

In [ ]:
dataset_path = "/content/drive/MyDrive/Houses Dataset"

Dataset k Andr Folder dekhna

In [ ]:
print(os.listdir(dataset_path)[:10])

['HousesInfo.txt', 'bathrooms', 'kitchens', 'frontals', 'bedrooms']


Dataset Read krna

In [ ]:

# text file load
columns = ["bedrooms","bathrooms","area","zipcode","price"]

df = pd.read_csv(dataset_path + "/HousesInfo.txt", sep=" ", header=None, names=columns)

df.head()

,bedrooms,bathrooms,area,zipcode,price
0,4,4.0,4053,85255,869500
1,4,3.0,3343,36372,865200
2,3,4.0,3923,85266,889000
3,5,5.0,4022,85262,910000
4,3,4.0,4116,85266,971226


Folder se Image Access krna

In [ ]:

bedroom_path = dataset_path + "/bedrooms"
bathroom_path = dataset_path + "/bathrooms"
frontal_path = dataset_path + "/frontals"
kitchen_path = dataset_path + "/kitchens"

print("Bedroom files:", os.listdir(bedroom_path)[:10])
print("Bathroom files:", os.listdir(bathroom_path)[:10])
print("Frontal files:", os.listdir(frontal_path)[:10])
print("Kitchen files:", os.listdir(kitchen_path)[:10])

Bedroom files: ['110_bedroom.jpg', '103_bedroom.jpg', '106_bedroom.jpg', '100_bedroom.jpg', '113_bedroom.jpg', '105_bedroom.jpg', '112_bedroom.jpg', '109_bedroom.jpg', '114_bedroom.jpg', '101_bedroom.jpg']
Bathroom files: ['111_bathroom.jpg', '103_bathroom.jpg', '102_bathroom.jpg', '101_bathroom.jpg', '106_bathroom.jpg', '115_bathroom.jpg', '109_bathroom.jpg', '100_bathroom.jpg', '108_bathroom.jpg', '113_bathroom.jpg']
Frontal files: ['103_frontal.jpg', '107_frontal.jpg', '104_frontal.jpg', '110_frontal.jpg', '108_frontal.jpg', '114_frontal.jpg', '100_frontal.jpg', '101_frontal.jpg', '102_frontal.jpg', '106_frontal.jpg']
Kitchen files: ['102_kitchen.jpg', '103_kitchen.jpg', '105_kitchen.jpg', '104_kitchen.jpg', '114_kitchen.jpg', '106_kitchen.jpg', '112_kitchen.jpg', '100_kitchen.jpg', '109_kitchen.jpg', '108_kitchen.jpg']


Image ko read krna and Array me Dalna

In [ ]:
# 🔹 Image size define kar rahe hain (har image ko same size me convert karna zaroori hota hai)
IMG_SIZE = 128

#  Different folders ke paths define kar rahe hain
bedroom_path = dataset_path + "/bedrooms"
bathroom_path = dataset_path + "/bathrooms"
frontal_path = dataset_path + "/frontals"
kitchen_path = dataset_path + "/kitchens"

#  Empty lists bana rahe hain jahan hum data store karenge
images = []         # combined images
tabular_data = []   # numeric features (bedrooms, bathrooms, area)
prices = []         # target variable

#  Har row (har house) ke liye loop chala rahe hain
for i in range(len(df)):

    try:
        #  Har house ke 4 images load kar rahe hain
        # filename pattern: 1_bedroom.jpg, 1_bathroom.jpg, etc.
        img1 = cv2.imread(f"{bedroom_path}/{i+1}_bedroom.jpg")
        img2 = cv2.imread(f"{bathroom_path}/{i+1}_bathroom.jpg")
        img3 = cv2.imread(f"{frontal_path}/{i+1}_frontal.jpg")
        img4 = cv2.imread(f"{kitchen_path}/{i+1}_kitchen.jpg")

        #  Agar koi image load nahi hui (None ayi), to us sample ko skip kar do
        if img1 is None or img2 is None or img3 is None or img4 is None:
            continue

        # Sab images ko same size (128x128) me resize kar rahe hain
        img1 = cv2.resize(img1,(IMG_SIZE,IMG_SIZE))
        img2 = cv2.resize(img2,(IMG_SIZE,IMG_SIZE))
        img3 = cv2.resize(img3,(IMG_SIZE,IMG_SIZE))
        img4 = cv2.resize(img4,(IMG_SIZE,IMG_SIZE))

        #  4 images ko horizontally combine kar rahe hain (side by side)
        combined_img = np.hstack([img1, img2, img3, img4])

        # Pixel values normalize kar rahe hain (0-255 → 0-1)
        combined_img = combined_img / 255.0

        #  Final image list me add kar rahe hain
        images.append(combined_img)

        #  Tabular features (numerical data) extract kar rahe hain
        tabular_data.append([
            df.iloc[i]["bedrooms"],
            df.iloc[i]["bathrooms"],
            df.iloc[i]["area"]
        ])

        #  Price (target value) store kar rahe hain
        prices.append(df.iloc[i]["price"])

    except:
        # Agar koi error aaye to us iteration ko skip kar do
        continue

#  Lists ko numpy arrays me convert kar rahe hain (model ke liye required hota hai)
images = np.array(images)
tabular_data = np.array(tabular_data)
prices = np.array(prices)

#  Final images ka shape print kar rahe hain
print("Images shape:", images.shape)

Images shape: (535, 128, 512, 3)


Train or Test ko Split krna


In [ ]:

X_img_train, X_img_test, X_tab_train, X_tab_test, y_train, y_test = train_test_split(
    images, tabular_data, prices, test_size=0.2, random_state=42
)

Image k Liye Layers setting

In [ ]:
image_input = Input(shape=(128,512,3)) # Changed input shape to (128, 512, 3)

x = Conv2D(32,(3,3),activation='relu')(image_input)
x = MaxPooling2D()(x)

x = Conv2D(64,(3,3),activation='relu')(x)
x = MaxPooling2D()(x)

x = Flatten()(x)

image_features = Dense(64,activation='relu')(x)

Tabular Data k liye Layer setting

In [ ]:
tabular_input = Input(shape=(3,))

y = Dense(32,activation='relu')(tabular_input)
y = Dense(16,activation='relu')(y)

Image or Tabular ko Combine krna

In [ ]:
# Image features aur tabular features (yahan 'y' actually tabular input hai)
# Dono ko combine kar rahe hain ek single vector me
combined = concatenate([image_features, y])

#  Fully connected (Dense) layer
# Ye layer combined features ko learn karti hai
z = Dense(32, activation='relu')(combined)

#  Final output layer   1 neuron use kiya gaya hai kyunki ye regression task hai (price predict karna hai)
output = Dense(1)(z)

Model for Tabular Data

In [ ]:
model = Model(inputs=[image_input, tabular_input], outputs=output)

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 128, 512,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 126, 510,  │        896 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 63, 255,   │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 61, 253,   │     18,496 │ max_pooling2d_2[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 30, 126,   │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 241920)    │          0 │ max_pooling2d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 32)        │        128 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 64)        │ 15,482,944 │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 16)        │        528 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 80)        │          0 │ dense_5[0][0],    │
│ (Concatenate)       │                   │            │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 32)        │      2,592 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 1)         │         33 │ dense_8[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 15,505,617 (59.15 MB)

 Trainable params: 15,505,617 (59.15 MB)

 Non-trainable params: 0 (0.00 B)

Model for Image

In [ ]:
model.fit(
    [X_img_train, X_tab_train],
    y_train,
    validation_data=([X_img_test, X_tab_test], y_test),
    epochs=10,
    batch_size=32
)

Epoch 1/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 14s 504ms/step - loss: 645021892608.0000 - mae: 597333.5000 - val_loss: 436990803968.0000 - val_mae: 546456.8750
Epoch 2/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - loss: 616532672512.0000 - mae: 571936.8125 - val_loss: 356389847040.0000 - val_mae: 468615.6250
Epoch 3/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - loss: 432549855232.0000 - mae: 419325.5312 - val_loss: 145331208192.0000 - val_mae: 283116.1562
Epoch 4/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - loss: 302187905024.0000 - mae: 375442.4375 - val_loss: 145805197312.0000 - val_mae: 284387.8750
Epoch 5/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 293670453248.0000 - mae: 326277.9375 - val_loss: 144210771968.0000 - val_mae: 280106.1562
Epoch 6/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - loss: 292716118016.0000 - mae: 348738.9688 - val_loss: 145159241728.0000 - val_mae: 283011.4062
Epoch 7/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - loss: 291739172864.0000 - mae: 322844.2188 - val_lo

RMSE & MEA find krna

In [ ]:
pred = model.predict([X_img_test, X_tab_test])

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print("MAE:", mae)
print("RMSE:", rmse)

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 181ms/step
MAE: 287678.117114486
RMSE: 383252.7253287255
